# 05 – Neural architectures under both protocols
Trains LSTM, GRU and CNN-LSTM with fixed settings and five seeds under the leakage-prone and sound protocols (date-matched, six walk-forward folds).

Set `QUICK_TEST = False` for the full run (about 50 minutes on a T4 GPU). Results are saved as `neural_protocols_rerun.csv`.

Final version: seeds 42–46 and a date-matched leakage-prone arm.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import os, time, random
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"

DEV_END = "2023-12-31"
TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]

gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", bool(gpus), gpus)
if not gpus:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU")

dev = {t: pd.read_csv(f"{FEATURES}/{t}_features.csv",
                      index_col=0, parse_dates=True).loc[:DEV_END]
       for t in TICKERS}

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"\n{len(TICKERS)} assets, {len(PREDICTORS)} predictors, "
      f"{len(dev['AAPL'])} rows each")

## 2. Experiment configuration
Held constant across every arm so that only the protocol varies: sequence length, architecture sizes, optimiser, learning rate, batch size, epochs, early-stopping patience.

In [ ]:
QUICK_TEST = True     # True: one asset, one seed. Set False for the full run.

SEQ_LEN    = 20       # trading days of history per sample (about one month)
SEEDS      = [42] if QUICK_TEST else [42, 43, 44, 45, 46]
ASSETS     = ["AAPL"] if QUICK_TEST else TICKERS

INITIAL_TRAIN = 1000
TEST_SIZE     = 250
STEP          = 250
N_NEURAL_FOLDS = 2 if QUICK_TEST else 6   # most recent folds only

EPOCHS     = 30
PATIENCE   = 5
BATCH_SIZE = 64
LR         = 1e-3
UNITS      = 32
DROPOUT    = 0.2

print(f"Mode: {'QUICK TEST' if QUICK_TEST else 'FULL RUN'}")
print(f"Assets: {ASSETS}")
print(f"Seeds:  {SEEDS}")
print(f"Neural walk-forward folds: {N_NEURAL_FOLDS}")
print(f"\nTotal training runs: "
      f"{len(ASSETS) * 3 * len(SEEDS) * (N_NEURAL_FOLDS + 1)}")

## 3. Sequence construction, and the overlap that causes the leak
Measures how much random-split test sequences overlap with the training data.

In [ ]:
def make_sequences(X, y, seq_len=SEQ_LEN):
    """Sliding windows. Sample i covers rows [i, i+seq_len) and predicts
    the target at row i+seq_len-1."""
    n = len(X) - seq_len + 1
    Xs = np.stack([X[i:i + seq_len] for i in range(n)])
    ys = y[seq_len - 1:]
    return Xs, ys, np.arange(seq_len - 1, len(X))   # original row of each sample


# --- quantify the overlap a random split creates -----------------------
d = dev["AAPL"]
X_demo = d[PREDICTORS].values
y_demo = d["target_direction"].values
Xs, ys, rows = make_sequences(X_demo, y_demo)

rng = np.random.default_rng(42)
perm = rng.permutation(len(Xs))
n_test = int(len(Xs) * 0.2)
test_i, train_i = perm[:n_test], perm[n_test:]

train_rows = set(rows[train_i])

# For each test sample, how many of its SEQ_LEN days also appear as the
# endpoint of some training sample?
overlaps = []
for i in test_i[:2000]:
    end = rows[i]
    window = range(end - SEQ_LEN + 1, end + 1)
    overlaps.append(sum(1 for r in window if r in train_rows))

overlaps = np.array(overlaps)

print(f"Sequence length: {SEQ_LEN} days")
print(f"Total samples:   {len(Xs)}")
print(f"Random split:    {len(train_i)} train / {len(test_i)} test\n")
print("For each test sequence, days also seen as a training endpoint:")
print(f"  mean:            {overlaps.mean():.1f} of {SEQ_LEN} days "
      f"({100*overlaps.mean()/SEQ_LEN:.0f}%)")
print(f"  minimum:         {overlaps.min()}")
print(f"  test sequences with >50% overlap: "
      f"{100*(overlaps > SEQ_LEN/2).mean():.1f}%")
print("\nUnder a random split, essentially every test sequence has near-")
print("duplicates in the training set. This is what a high-capacity model")
print("can exploit and a 20-coefficient linear model cannot.")

## 4. Architectures
Small networks (32 units) to suit the size of the data.

In [ ]:
def set_seeds(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)


def build_lstm(shape):
    return keras.Sequential([
        layers.Input(shape=shape),
        layers.LSTM(UNITS),
        layers.Dropout(DROPOUT),
        layers.Dense(1, activation="sigmoid"),
    ], name="LSTM")


def build_gru(shape):
    return keras.Sequential([
        layers.Input(shape=shape),
        layers.GRU(UNITS),
        layers.Dropout(DROPOUT),
        layers.Dense(1, activation="sigmoid"),
    ], name="GRU")


def build_cnn_lstm(shape):
    return keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv1D(32, kernel_size=3, activation="relu", padding="causal"),
        layers.MaxPooling1D(2),
        layers.LSTM(UNITS),
        layers.Dropout(DROPOUT),
        layers.Dense(1, activation="sigmoid"),
    ], name="CNN-LSTM")


ARCHITECTURES = {"LSTM": build_lstm, "GRU": build_gru, "CNN-LSTM": build_cnn_lstm}

shape = (SEQ_LEN, len(PREDICTORS))
for name, fn in ARCHITECTURES.items():
    m = fn(shape)
    print(f"{name:<10} {m.count_params():>7,} parameters")

print(f"\nLogistic regression, for comparison: {len(PREDICTORS) + 1} parameters")

## 5. Training harness
Sound protocol: scaler fitted on training rows only; the first test sequences use the last 19 training days as inputs only.

In [ ]:
def train_and_predict(build_fn, Xtr, ytr, Xte, seed):
    """Train one model, return binary predictions on the test set."""
    set_seeds(seed)
    model = build_fn(Xtr.shape[1:])
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="binary_crossentropy", metrics=["accuracy"])

    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                         restore_best_weights=True)

    model.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.15, callbacks=[stop], verbose=0)

    return (model.predict(Xte, verbose=0).ravel() > 0.5).astype(int)


def neural_test_window(df, n_folds):
    """First and last row of the test period covered by the sound arm's folds."""
    all_folds, start = [], INITIAL_TRAIN
    while start + TEST_SIZE <= len(df):
        all_folds.append((start, start + TEST_SIZE))
        start += STEP
    folds = all_folds[-n_folds:]
    return folds[0][0], folds[-1][1]


def protocol_leakage_prone(df, build_fn, seed, n_folds=None):
    """Global scaling, sequences built across the whole series, random split.

    Date-matched to the sound arm: the test set is every sequence whose target
    falls in the same test window as the sound arm's folds, so both protocols
    are scored on identical dates and sample sizes. The remaining sequences,
    from before and after that window, are shuffled and used for training.
    """
    n_folds = N_NEURAL_FOLDS if n_folds is None else n_folds
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    X = StandardScaler().fit_transform(X)          # leak 1: global statistics
    Xs, ys, rows = make_sequences(X, y)

    first_test, last_test = neural_test_window(df, n_folds)
    in_window = (rows >= first_test) & (rows < last_test)

    rng = np.random.default_rng(seed)
    te = np.where(in_window)[0]                    # same dates as the sound arm
    tr = rng.permutation(np.where(~in_window)[0])  # leak 2: shuffled windows,
                                                   # including later observations
    pred = train_and_predict(build_fn, Xs[tr], ys[tr], Xs[te], seed)
    return ys[te], pred


def protocol_sound(df, build_fn, seed, n_folds):
    """Fold-fitted scaling, expanding walk-forward, purged fold boundaries."""
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    all_folds, start = [], INITIAL_TRAIN
    while start + TEST_SIZE <= len(df):
        all_folds.append((start, start + TEST_SIZE))
        start += STEP
    folds = all_folds[-n_folds:]

    y_true, y_pred = [], []

    for split, end in folds:
        scaler = StandardScaler().fit(X[:split])   # training rows only
        Xsc = scaler.transform(X)

        Xtr, ytr, _ = make_sequences(Xsc[:split], y[:split])

        # purge: test sequences start far enough in that none reuse training rows
        Xte, yte, _ = make_sequences(Xsc[split - SEQ_LEN + 1:end],
                                     y[split - SEQ_LEN + 1:end])

        pred = train_and_predict(build_fn, Xtr, ytr, Xte, seed)
        y_pred.extend(pred)
        y_true.extend(yte)

    return np.array(y_true), np.array(y_pred)


print("Harness ready - leakage-prone arm now date-matched to sound arm")

## 6. Run the experiment

In [ ]:
records = []
t_start = time.time()
total = len(ASSETS) * len(ARCHITECTURES) * len(SEEDS) * 2
done = 0

for ticker in ASSETS:
    df = dev[ticker]

    for arch_name, build_fn in ARCHITECTURES.items():
        for seed in SEEDS:

            for protocol, fn in [
                ("leakage_prone", lambda: protocol_leakage_prone(df, build_fn, seed)),
                ("sound",         lambda: protocol_sound(df, build_fn, seed, N_NEURAL_FOLDS)),
            ]:
                yt, yp = fn()

                acc = accuracy_score(yt, yp)
                records.append({
                    "ticker": ticker, "architecture": arch_name,
                    "protocol": protocol, "seed": seed,
                    "n": len(yt),
                    "accuracy": round(acc, 4),
                    "mcc": round(matthews_corrcoef(yt, yp), 4),
                    "pred_up_rate": round(yp.mean(), 4),
                })

                done += 1
                print(f"[{done:>3}/{total}] {ticker:<5} {arch_name:<9} "
                      f"{protocol:<14} seed {seed}  acc {acc:.4f}  "
                      f"({time.time()-t_start:.0f}s)")

neural = pd.DataFrame(records)
print(f"\nComplete in {time.time()-t_start:.0f}s")

suffix = "_quick" if QUICK_TEST else "_rerun"   # never overwrites the original results
neural.to_csv(f"{RESULTS}/neural_protocols{suffix}.csv", index=False)

## 7. Results and the capacity comparison

In [ ]:
summary = (neural.groupby(["architecture", "protocol"])
                 .agg(accuracy_mean=("accuracy", "mean"),
                      accuracy_sd=("accuracy", "std"),
                      mcc_mean=("mcc", "mean"))
                 .round(4))

print("Mean across assets and seeds\n")
print(summary.to_string())

print("\n\nProtocol inflation by architecture\n")

infl = []
for arch in ARCHITECTURES:
    lp = neural.query("architecture == @arch and protocol == 'leakage_prone'")["accuracy"].mean()
    sd = neural.query("architecture == @arch and protocol == 'sound'")["accuracy"].mean()
    infl.append({"model": arch,
                 "leakage_prone": round(lp, 4),
                 "sound": round(sd, 4),
                 "inflation_pts": round(100 * (lp - sd), 2)})

# from notebook 03, for the capacity contrast
infl.append({"model": "LogisticRegression",
             "leakage_prone": 0.5137, "sound": 0.5115, "inflation_pts": 0.23})

inflation = pd.DataFrame(infl).sort_values("inflation_pts", ascending=False)
print(inflation.to_string(index=False))

print("\nIf the neural inflation figures substantially exceed 0.23, the")
print("capacity hypothesis is supported: leakage inflates flexible models")
print("more than rigid ones.")

inflation.to_csv(f"{RESULTS}/capacity_comparison{suffix}.csv", index=False)

In [ ]:
print("Seed variability under the sound protocol")
print("(spread across seeds shows how misleading a single run would be)\n")

sound_only = neural.query("protocol == 'sound'")

spread = (sound_only.groupby(["ticker", "architecture"])["accuracy"]
                    .agg(["min", "mean", "max", "std"])
                    .round(4))
print(spread.to_string())

if len(SEEDS) > 1:
    print(f"\nLargest within-configuration range across seeds: "
          f"{(spread['max'] - spread['min']).max():.4f}")
    print("A paper reporting one seed could have reported any value in that range.")

print("\n\nPrediction rate check - is a model just predicting one class?\n")
print(neural.groupby(["architecture", "protocol"])["pred_up_rate"]
            .agg(["mean", "min", "max"]).round(3).to_string())
print("\nValues near 1.0 or 0.0 mean the model collapsed to a constant,")
print("which is the neural equivalent of the majority-class baseline.")

## 8. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 05: neural architectures"
             f"{' (QUICK TEST)' if QUICK_TEST else ''}")
lines.append("")
lines.append("**Configuration**")
lines.append(f"- Sequence length: {SEQ_LEN} days")
lines.append(f"- Architectures: LSTM, GRU, CNN-LSTM ({UNITS} units, dropout {DROPOUT})")
lines.append(f"- Seeds: {SEEDS}")
lines.append(f"- Assets: {ASSETS}")
lines.append(f"- Walk-forward folds used: {N_NEURAL_FOLDS} (most recent)")
lines.append(f"- Fixed hyperparameters across all arms; no per-arm tuning")
lines.append("")
lines.append("**Sequence overlap under random splitting**")
lines.append(f"- Mean overlap between a test sequence and training endpoints: "
             f"{overlaps.mean():.1f} of {SEQ_LEN} days "
             f"({100*overlaps.mean()/SEQ_LEN:.0f}%)")
lines.append(f"- Test sequences with more than 50% overlap: "
             f"{100*(overlaps > SEQ_LEN/2).mean():.1f}%")
lines.append("")
lines.append("**Results**")
lines.append("")
lines.append(summary.to_string())
lines.append("")
lines.append("**Protocol inflation by model capacity**")
lines.append("")
lines.append(inflation.to_string(index=False))
lines.append("")
lines.append("**Seed variability, sound protocol**")
lines.append("")
lines.append(spread.to_string())
lines.append("")
lines.append("**Next:** Notebook 06 - significance testing "
             "(Clark-West for nested comparisons, Diebold-Mariano for non-nested, "
             "moving-block bootstrap CIs, FDR correction)")

entry = "\n".join(lines)

with open(f"{BASE}/results_log_rerun.md", "a") as f:
    f.write(entry)

print(entry)